# Train the glyph segmentation detector (Colab, GPU)

Run this notebook on Colab with a GPU runtime
(Runtime -> Change runtime type -> GPU).

This fine-tunes a pretrained YOLOv8-segmentation checkpoint (see
`reports/2026-09-15-segmentation-detector-sourcing.md`) on synthetic
composite images built from our own single-glyph crops
(`data/raw/`) -- see `docs/superpowers/specs/2026-09-15-glyph-segmentation-detector-design.md`
for the full design.

## 1. Clone the repo and install it

In [ ]:
!git clone https://github.com/DelfinEryilmaz/hieroglyph-translator.git
%cd hieroglyph-translator
!pip install -e . -q
!pip install ultralytics -q

## 2. Check GPU is available

In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected -- check Runtime > Change runtime type > GPU")

## 3. Upload the glyph crop dataset

Same dataset the classifier trains on. Upload the original `archive.zip`
from Kaggle (or a zip of your local `data/raw/` folder) -- either works,
this cell handles both.

In [ ]:
from google.colab import files

uploaded = files.upload()  # pick archive.zip (from Kaggle) or your own data_raw.zip
uploaded_filename = next(iter(uploaded))

In [ ]:
import shutil
import zipfile
from pathlib import Path

DATA_ROOT = Path("data/raw")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(uploaded_filename) as zf:
    zf.extractall(DATA_ROOT)


def class_folders(root):
    return [p for p in root.iterdir() if p.is_dir()]


top_level = class_folders(DATA_ROOT)
# If the zip had one extra wrapper folder (some Windows re-zips do this),
# flatten it so DATA_ROOT/<class>/*.png is the actual layout.
if len(top_level) == 1 and not list(top_level[0].glob("*.png")):
    inner = top_level[0]
    for child in inner.iterdir():
        shutil.move(str(child), str(DATA_ROOT / child.name))
    inner.rmdir()

print(f"{len(class_folders(DATA_ROOT))} class folders under {DATA_ROOT}")

## 4. Split crop files, then generate synthetic composites

Splitting happens on individual *crop files* before compositing, not on
composites -- so no single glyph crop's pixels ever appear in both the
train and test splits (see `hieroglyph.segmentation.synthesize.split_crop_paths`'s
docstring).

In [ ]:
from pathlib import Path
from hieroglyph.segmentation.synthesize import generate_dataset, list_crop_paths, split_crop_paths

RAW_DIR = Path("data/raw")
SYNTH_DIR = Path("data/synthetic_composites")

crop_paths = list_crop_paths(RAW_DIR)
train_crops, valid_crops, test_crops = split_crop_paths(crop_paths, seed=0)
print(f"{len(train_crops)} train crops, {len(valid_crops)} valid crops, {len(test_crops)} test crops")

generate_dataset(train_crops, SYNTH_DIR / "train", num_composites=4000, seed=0)
generate_dataset(valid_crops, SYNTH_DIR / "valid", num_composites=500, seed=1)
generate_dataset(test_crops, SYNTH_DIR / "test", num_composites=500, seed=2)
print("done generating composites")

In [ ]:
import yaml

data_yaml = {
    "path": str(SYNTH_DIR.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": {0: "hieroglyph"},
    "nc": 1,
}
(SYNTH_DIR / "data.yaml").write_text(yaml.dump(data_yaml), encoding="utf-8")
print((SYNTH_DIR / "data.yaml").read_text())

## 5. Download the pretrained checkpoint we're fine-tuning from

In [ ]:
!python scripts/download_pretrained_yolo.py --dest models/yolo_seg_pretrained.pt

## 6. Fine-tune

In [ ]:
from ultralytics import YOLO

model = YOLO("models/yolo_seg_pretrained.pt")
train_results = model.train(
    data=str(SYNTH_DIR / "data.yaml"),
    epochs=30,
    imgsz=640,
    device=0,
    project="runs",
    name="yolo_seg_finetune",
)

## 7. Download the fine-tuned checkpoint

This is the file that goes into your local `models/yolo_seg.pt` for
`notebooks/05_evaluate_segmenter.ipynb` and the Streamlit demo (Phase 12).
**Rename it to `yolo_seg.pt` locally** after downloading -- both of those
expect that exact filename.

In [ ]:
from google.colab import files

files.download("runs/yolo_seg_finetune/weights/best.pt")